# JS10 Lab 3: Klasifikasi Churn Rate dengan ANN

Jobsheet 10, Lab 3. Dataset `dataset/Churn_Modelling.csv` berisi 10.000 nasabah bank. Target `Exited` bernilai 1 kalau nasabah berhenti menjadi customer. Model ANN dibangun dengan TensorFlow/Keras.

## Pra Pengolahan Data

### Langkah 1 - Import Library

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # sembunyikan log TensorFlow

import numpy as np
import pandas as pd
import tensorflow as tf

tf.keras.utils.set_random_seed(0)  # hasil bisa berbeda tiap run tanpa seed

### Langkah 2 - Load Data

In [2]:
dataset = pd.read_csv('dataset/Churn_Modelling.csv')
X = dataset.iloc[:, 3:-1].values
y = dataset.iloc[:, -1].values

In [3]:
print(X)

[[619 'France' 'Female' ... 1 1 101348.88]
 [608 'Spain' 'Female' ... 0 1 112542.58]
 [502 'France' 'Female' ... 1 0 113931.57]
 ...
 [709 'France' 'Female' ... 0 1 42085.58]
 [772 'Germany' 'Male' ... 1 0 92888.52]
 [792 'France' 'Female' ... 1 0 38190.78]]


Kolom `RowNumber`, `CustomerId`, dan `Surname` dibuang lewat irisan `3:-1` karena tidak ada hubungannya dengan churn. Sisanya sepuluh fitur: CreditScore, Geography, Gender, Age, Tenure, Balance, NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary.

### Langkah 3 - Encoding Data Kategorikal

In [4]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:, 2])

In [5]:
print(X)

[[619 'France' 0 ... 1 1 101348.88]
 [608 'Spain' 0 ... 0 1 112542.58]
 [502 'France' 0 ... 1 0 113931.57]
 ...
 [709 'France' 0 ... 0 1 42085.58]
 [772 'Germany' 1 ... 1 0 92888.52]
 [792 'France' 0 ... 1 0 38190.78]]


### Langkah 4 - Encoding Kolom "Geography" dengan One Hot Encoder

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [1])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

In [7]:
print(X)

[[1.0 0.0 0.0 ... 1 1 101348.88]
 [0.0 0.0 1.0 ... 0 1 112542.58]
 [1.0 0.0 0.0 ... 1 0 113931.57]
 ...
 [1.0 0.0 0.0 ... 0 1 42085.58]
 [0.0 1.0 0.0 ... 1 0 92888.52]
 [1.0 0.0 0.0 ... 1 0 38190.78]]


Geography berisi tiga negara dan tidak punya urutan, jadi dipecah jadi tiga kolom 0/1. Gender cukup LabelEncoder karena hanya dua nilai. Setelah langkah ini `X` punya 12 kolom, sesuai urutan yang nanti dipakai saat prediksi: France, Germany, Spain, CreditScore, Gender, Age, Tenure, Balance, NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary.

### Langkah 5 - Split Data

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

### Langkah 6 - Scaling Fitur

In [9]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

Scaler di-fit hanya pada data latih. Kalau `fit_transform` dipakai di data uji juga, informasi data uji ikut bocor ke pelatihan. Rentang Balance dan EstimatedSalary jauh lebih besar dari kolom 0/1, dan ANN sensitif terhadap skala seperti ini.

## Membuat Model ANN

### Langkah 1 - Inisiasi Model ANN

In [10]:
ann = tf.keras.models.Sequential()

### Langkah 2 - Membuat Input Layer dan Hidden Layer Pertama

In [11]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

### Langkah 3 - Membuat Hidden Layer Kedua

In [12]:
ann.add(tf.keras.layers.Dense(units=6, activation='relu'))

### Langkah 4 - Membuat Output Layer

In [13]:
ann.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

Kedua hidden layer memakai ReLU, output memakai sigmoid karena targetnya biner. Satu neuron output sudah cukup: nilainya probabilitas kelas 1, dan kelas 0 adalah sisanya.

## Training Model

### Langkah 1 - Compile Model (Menyatukan Arsitektur) ANN

In [14]:
ann.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

### Langkah 2 - Fitting Model

In [15]:
ann.fit(X_train, y_train, batch_size = 32, epochs = 100, verbose = 0)

## Membuat Prediksi

Data nasabah baru:

* Geography: France
* Credit Score: 600
* Gender: Male
* Age: 40 years old
* Tenure: 3 years
* Balance: 60000 USD
* Number of Products: 2
* Does this customer have a credit card? Yes
* Is this customer an Active Member: Yes
* Estimated Salary: 50000 USD

### Modelkan Data Baru dan Buat Prediksi

In [16]:
print(ann.predict(sc.transform([[1, 0, 0, 600, 1, 40, 3, 60000, 2, 1, 1, 50000]]), verbose = 0) > 0.5)

[[False]]


Hasilnya `False`, artinya probabilitas churn di bawah 0.5. Nasabah ini diprediksi bertahan, jadi belum perlu diprioritaskan untuk program retensi.

### Prediksi Dengan Data Testing

In [17]:
y_pred = ann.predict(X_test, verbose = 0)
y_pred = (y_pred > 0.5)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[0 0]
 [0 1]
 [0 0]
 ...
 [0 0]
 [0 0]
 [0 0]]


### Cek Akurasi dan Confusion Matrix

In [18]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[1507   88]
 [ 194  211]]


0.859

Akurasi 0.859 terlihat tinggi, tetapi confusion matrix menunjukkan sisi lain. Dari 405 nasabah yang benar-benar churn, 194 (sekitar 48%) diprediksi bertahan. Artinya setengah calon churn lolos dari deteksi.

Angka ini konsisten dengan komposisi data: sekitar 80% baris berlabel 0, jadi menebak 0 untuk semua data pun sudah memberi akurasi 0.80. Akurasi saja tidak cukup untuk menilai model pada data tidak seimbang; recall kelas churn yang perlu dilihat. Kalau tujuan bisnisnya menangkap calon churn, bobot kelas minoritas perlu dinaikkan atau ambang 0.5 diturunkan, dengan konsekuensi lebih banyak nasabah setia ikut ditandai berisiko.